# Unit 17: Workspace Ingestion and Guardrail Orchestration

## Introduction

Welcome to **Applying What You've Learned**! This is the first lesson of the course, and we are starting with something fundamental: configuring Claude Code to work safely and effectively within an existing repository workspace.

When deploying Claude Code onto a real-world enterprise codebase, you cannot simply jump in and permit the agent to make unconstrained modifications. The language model must first ingest and internalize the project's layout topology, internal syntax conventions, and safe execution boundaries. Think of it exactly like onboarding a new software engineer: before writing patches, they must learn the structure, understand the testing thresholds, and recognize which assets are strictly off-limits. That is exactly what we will accomplish in this module.

By the end of this unit, we will have fully configured Claude Code for an Express/SQLite Todo API project, instantiating the three foundational pillars of workspace management: `CLAUDE.md` for orchestrating project standards, `.claudeignore` for context-window optimization, and `settings.json` for deterministic permission gating. We will then verify the integrity of the setup through trace logging.

---

## The Exploration Phase: Codebase Discovery

Before writing a single line of configuration metadata, you must execute a structural survey of the repository. This discovery phase is critical because your guardrails and instruction files must accurately mirror the structural logic and code styles already established within the system.

When Claude Code mounts a fresh workspace, it exposes a suite of high-performance native filesystem utilities to scan the repository tree. The primary tools deployed during ingestion include:

* **`Bash` / `Glob`:** For mapping directory layouts and exploring file trees.
* **`Read`:** For inspecting code files to analyze coding patterns.
* **`Grep`:** For scanning keyword distribution across the code blocks.

By running this upfront discovery pass, we isolate five critical metrics: the underlying technology stack, directory organization, functional syntax patterns, error-handling contracts, and the test suite runner configuration.

---

## Mapping Project Topology

Let's begin by tracing the directory structural layout of our target Todo API codebase. We can instruct Claude to map out the folder trees via a recursive directory scan:

```bash
ls -R src/

```

The system stdout exposes the following modular organization:

```text
src/
├── server.js
├── routes/
│   ├── todos.js
│   └── users.js
├── middleware/
│   ├── auth.js
│   └── validate.js
└── models/
    └── Todo.js

```

### Key Structural Insights Isolated:

* **Strict Layer Separation:** The architecture enforces a classic MVC-style division, decoupling route parameters from middleware logic and database model schemas.
* **Feature-Sliced Routing:** Individual application domains (such as `/todos` and `/users`) are broken out into separate feature modules.
* **Interceptors Directory:** A dedicated `middleware/` folder indicates that reusable global routines (like session token evaluation or request schemas) are abstracted away from core business endpoints.

---

## Identifying Code Patterns and Contracts

To ensure Claude writes seamless, native patches that match your existing design patterns, we must audit the file syntax profiles across three distinct architectural layers.

### 1. Main Entrypoint Integration (`src/server.js`)

```javascript
// Express app with JWT auth, SQLite database
const express = require('express');
const jwt = require('jsonwebtoken');
const Database = require('better-sqlite3');
const app = express();
// ... setup code

```

### 2. Service Routing Tier (`src/routes/todos.js`)

```javascript
// Standard CRUD operations, middleware-based validation
router.get('/', authenticateToken, async (req, res) => {
  try {
    const todos = await getTodos(req.user.id);
    res.json({ data: todos });
  } catch (error) {
    res.status(500).json({ error: error.message });
  }
});

```

### 3. Integration Testing Suite (`tests/todo.test.js`)

```javascript
const request = require('supertest');
const app = require('../src/server');

describe('Todo API', () => {
  test('GET /api/todos returns todos', async () => {
    const res = await request(app)
      .get('/api/todos')
      .set('Authorization', `Bearer ${authToken}`);
        
    expect(res.status).toBe(200);
  });
});

```

### 🔍 Foundational Code Style Invariants:

* **Asynchronous Control Flow:** All database pipelines and routing hooks strictly leverage `async/await` syntax instead of old-school callback patterns.
* **Unified Output Schema:** Success payloads are always wrapped inside a root `{ data: {...} }` object, while error handlers serialize exceptions into a standardized `{ error: "message" }` JSON wire format.
* **Decoupled Verification Testing:** The testing architecture relies on the **Jest** runner coupled with **Supertest** HTTP assertion assertions, requiring explicit header authorization tokens.

---

## Constructing `CLAUDE.md`: The System Operating Manual

With our workspace conventions fully mapped, we can now write the `CLAUDE.md` file. This document acts as the primary instruction layer that Claude Code reads during session initialization to guide all code generation tasks.

### Complete Document Layout: `CLAUDE.md`

Create this file in the root of your project directory:

```markdown
# Todo API Project

## Overview
REST API for todo management with user authentication

## Tech Stack
- Node.js 20+
- Express 4.x
- SQLite with better-sqlite3
- JWT authentication
- Jest for testing

## Coding Standards
- Use async/await syntax exclusively (no legacy callbacks)
- Extract request validation routines into dedicated middleware modules (`src/middleware/`)
- Enforce standardized success response wire envelopes: `{ data: {...} }`
- Enforce standardized error response wire envelopes: `{ error: "message" }`
- All route actions must be wrapped in try/catch blocks for explicit error handling
- Document all function blocks with descriptive JSDoc commentary arrays

## Testing
- Execution command: `npm test`
- Framework environment: Jest paired with supertest
- Minimum strict test coverage threshold: 80%

## Security
- Never leak, print, or log authentication tokens or user passwords to stdout
- Validate and sanitize all incoming request parameters before query compilation
- Prevent injection attacks by writing parameterized queries exclusively

```

By explicitly locking down dependency versions, testing commands, and error response envelopes within `CLAUDE.md`, you prevent the agent from introducing mismatched paradigms or proposing insecure code implementations.

---

## Context Window Optimization with `.claudeignore`

Just as a `.gitignore` file stops build artifacts from polluting your Git history, a `.claudeignore` file prevents unnecessary files from cluttering Claude's active context window. Excluding non-source data is crucial for maximizing execution speed, minimizing token costs, and protecting project data security.

Create a `.claudeignore` file in your repository root and apply this production configuration:

```text
# Dependencies
node_modules/

# Test artifacts
coverage/
*.test.db

# Logs
*.log
logs/

# Sensitive data
.env
*.key
*.pem
secrets/

```

### Deconstructing the Mitigation Rules:

* **`node_modules/`:** Prevents the system from scanning thousands of dependency files, conserving valuable context space.
* **`coverage/` and `*.test.db`:** Excludes volatile, machine-generated artifacts that overwrite frequently during test executions.
* **`*.log`:** Keeps messy runtime diagnostic printouts from crowding out clean source files.
* **`.env`, `*.key`, `*.pem`, `secrets/`:** **Critical security boundary.** Hard-gating credentials and environment secrets stops sensitive cryptographic assets from being accidentally ingested into the model's memory buffer.

---

## Configuring Sandbox Tool Permissions

To establish deterministic permission guardrails under the principle of least privilege, we define clear access boundaries within the `.claude/settings.json` file. This prevents the model from mutating local resources without explicit oversight.

Create the file at `.claude/settings.json` and deploy this security matrix:

```json
{
  "model": "claude-sonnet-4-5",
  "permissions": {
    "allow": ["Read", "WebSearch"],
    "ask": ["Edit", "Write", "Bash"]  }
}

```

### Evaluating the Security Profiles:

* **`allow: ["Read", "WebSearch"]`:** Auto-approves information gathering. This speeds up exploration loops by allowing Claude to read files and search documentation without constantly pausing for manual approval.
> ⚠️ **Security Advisory:** Auto-approving `Read` assumes your `.claudeignore` file is perfectly configured to mask sensitive data. If you are handling proprietary systems or trade secrets, consider shifting `WebSearch` to `"ask"` to prevent external queries from leaking project metrics to public search indexes.


* **`ask: ["Edit", "Write", "Bash"]`:** Enforces strict manual oversight. Any tool call that changes code lines or spawns local shell subprocesses stops execution until you explicitly authorize the action.

---

## Verifying Guardrail Integrity via Execution Logs

To verify that your workspace configurations and permission sets are properly aligned, execute a simple architectural query at your terminal prompt:

```text
> Now tell me about the authentication system in this project

```

Audit your terminal stdout logging stream to observe how the configuration parameters guide the tool runtime:

```text
● Read(src/middleware/auth.js)
  [Auto-approved - no prompt!]
  ⎿ Read 25 lines

● This project uses JWT authentication:
  
  - authenticateToken middleware verifies JWT
  - Token expected in Authorization header: "Bearer <token>"
  - Token contains user ID
  - Protected routes use this middleware
  - Tokens generated at login (/api/users/login)
  
  Following your CLAUDE.md standards, the middleware:
  - Returns error format: { error: "message" }
  - Uses async/await
  - Includes proper error handling

```

### Post-Execution Verification Audit:

1. **Frictionless Ingestion:** The `Read` tool ran instantly without displaying an interactive permission prompt, confirming that your `settings.json` allowlist is working perfectly.
2. **Context Alignment:** Claude accurately aligned its architectural summary with the specific standards declared inside your `CLAUDE.md` handbook, explicitly tracking your async/await and JSON error-envelope mandates.

---

## Conclusion

By configuring these three core workspace files, you transform Claude Code from a generic coding assistant into a well-integrated development partner. The `CLAUDE.md` handbook ensures architectural consistency, `.claudeignore` filters out context noise while protecting secrets, and `settings.json` enforces safe permission guardrails.

Let's head into the hands-on practice labs to implement and test these workspace security rules across varying project setups!

## Exploring an Unfamiliar Codebase with Claude

It's time to put Claude Code to work on a real project! In this exercise, you will start with an unconfigured Todo API and learn how Claude can help you understand an unfamiliar codebase before you make any changes.

Your first step is to ask Claude to explore the project structure. Request that Claude use its Bash tool to list the directories and then use the Read tool to examine key files, such as the main server file, route handlers, and any configuration files it finds. Let Claude become familiar with what is there.

Once Claude has explored the codebase, have a conversation in which you ask it to identify the main coding patterns it discovered. Specifically, ask about:

    How the code uses async/await
    The format that errors follow
    What middleware patterns are present

Finally, ask Claude to create a basic CLAUDE.md file that documents just two sections: an overview of the project and the tech stack it uses. This file will be based on what Claude learned during its exploration.

By the end of this exercise, you will have experienced how Claude can analyze a project and create helpful documentation — a skill that makes working with any codebase much easier!

To ingest and map out an unfamiliar codebase using Claude Code, we will run an exploratory survey, extract core architectural contracts, and initialize your project's `CLAUDE.md` handbook.

Execute this step-by-step input loop inside your interactive terminal session:

### Step 1: Initialize Codebase Structural Discovery

Type this broad discovery command at your `>` prompt and press **Enter**:

```text
Explore the project directory structure using your tools, map out the layout under src/, and examine the primary entrypoint files to understand how the application initializes.

```

* **Observe the Process:** Claude will deploy its native `Bash`, `Glob`, or `Read` tools to scan the folder layouts. Watch it ingest files like `src/server.js` or route handlers in the background to build its internal model of your architecture.

---

### Step 2: Extract Coding Invariants and Contracts

Once the initial exploration trace completes, prompt Claude to isolate the functional patterns hidden within the code logic:

```text
Identify the core coding patterns established across the codebase. Detail how the project utilizes async/await, the specific JSON wire format that error responses follow, and what middleware interceptors are present.

```

* **Verify:** Ensure Claude accurately pulls out your implicit architectural conventions, such as whether routes return standardized `{ error: "message" }` blocks or wrap success objects in data envelopes.

---

### Step 3: Scaffold the `CLAUDE.md` System Manual

Now, instruct Claude to generate the foundation of your operating manual using the parameters it just discovered. Enter this command:

```text
Create a basic CLAUDE.md file in the root directory containing exactly two foundational markdown sections: a concise '# Overview' of the project's purpose and a structured '## Tech Stack' list detailing the language runtimes and core packages.

```

* **Authorize:** When the file creation panel shows the proposed markdown structure diff on your screen, select **`1. Yes`** to approve writing the document to your workspace disk.

---

### Step 4: Finalize and Submit

With the structure explored, coding patterns successfully mapped, and the initial `CLAUDE.md` baseline safely written to your root directory, conclude this workspace onboarding module by typing:

```text
/submit

```

By completing this exercise, you have successfully automated the codebase onboarding process—leveraging Claude to rapidly ingest an unfamiliar repository layout and document its operational core! 🚀

## Building a Complete Configuration Setup

Now that Claude has explored the Todo API and created initial documentation, it's time to build a complete configuration that will make working together smooth and efficient!

In this exercise, you'll create the full three-file setup that professional developers use when working with Claude Code. Start by asking Claude to expand your CLAUDE.md file to include all the important sections: Overview, Tech Stack, Coding Standards, Testing, and Security guidelines. Claude should base these sections on the patterns it discovered while exploring the codebase in the previous exercise.

Next, work with Claude to create a .claudeignore file. This file tells Claude which parts of the project to skip when reading or searching. Ask Claude to exclude:

    node_modules/ (dependencies don't need review)
    Test artifacts like coverage/ and *.test.db
    Log files that change frequently

Finally, configure the .claude/settings.json file to control which tools Claude can use automatically versus which ones need your approval. Set it up so Claude can auto-approve Read and WebSearch permissions (these are safe and speed things up), but must ask for confirmation before using Edit, Write, or Bash commands (these make changes, so you want control).

By completing this configuration, you'll have a fully optimized workspace where Claude can help efficiently while keeping you in control of important decisions!

To finalize your repository guardrails, we will engineer the full three-file configuration matrix. This setup optimizes Claude Code's context processing speed, enforces your project's architectural invariants, and locks down execution sandboxes under the principle of least privilege.

Execute this step-by-step input loop inside your active terminal session to build out your workspace configurations:

### Step 1: Expand `CLAUDE.md` into a Comprehensive Operating Manual

Copy and paste this instruction into your `>` prompt and press **Enter**:

```text
Expand the CLAUDE.md file in the root directory to include five distinct sections: '# Overview', '## Tech Stack', '## Coding Standards', '## Testing', and '## Security'. Base these sections explicitly on the async/await logic, JSON error response shapes, middleware structures, and Jest test runner configurations identified during exploration.

```

* **Authorize:** When the file patch diff panels display the extended markdown layout, select **`1. Yes`** to approve writing the updates to your disk.

---

### Step 2: Configure Context Isolation via `.claudeignore`

Now, construct your filesystem filter to strip out runtime noise and shield sensitive environment assets. Type this directive and press **Enter**:

```text
Create a .claudeignore file in the root directory to mask non-source files. Exclude node_modules/, coverage/ artifacts, *.test.db files, logs/ or *.log paths, and sensitive secret tokens or environment files like .env, *.key, or *.pem.

```

* **Authorize:** Select **`1. Yes`** when the file creation diff renders to commit the ignore filters to your workspace.

---

### Step 3: Establish Sandbox Tool Boundaries via `settings.json`

To build a deterministic security gateway that permits frictionless information harvesting while mandating explicit manual validation for system modifications, write your rule configurations by entering this prompt:

```text
Create a settings.json file inside the .claude/ directory. Configure the permissions matrix object to auto-approve "Read" and "WebSearch" actions under the "allow" array, while placing state-mutating tools—specifically "Edit", "Write", and "Bash"—inside the "ask" array to mandate strict confirmation dialogs.

```

* **Authorize:** When the structured JSON file block appears on your screen, select **`1. Yes`** to lock down the permissions engine.

---

### Step 4: Verify the Complete Multi-Guardrail Alignment

With all three configuration pieces written to disk, verify that your rule intersections function seamlessly together. Ask Claude an analytical query to watch the permission gates handle the task:

```text
Explain how the current middleware system processes a request payload, and highlight how your answer aligns with our CLAUDE.md guidelines.

```

* **Verify the Trace:** Look closely at the background execution log steps. Notice that the **`Read`** tool fires instantly and executes with *zero* confirmation prompts because it is explicitly allowed in your settings. Furthermore, verify that Claude frames its analytical breakdown using the exact async parameters and JSON error structures mandated by your updated `CLAUDE.md` handbook!

---

### Step 5: Finalize and Submit

Once the three-file configuration matrix is verified via execution logs and fully captured in your workspace session trace, close out this setup module by typing:

```text
/submit

```

By completing this exercise, you have built a fully optimized, secure developer workspace where Claude Code can run high-performance background diagnostics while keeping you in absolute control of all file alterations and command executions! 🚀

## Testing Your Configuration in Action

Time to see your configuration in action! Now that you've set up all three configuration files, you'll verify that everything works as intended by testing both the permission system and Claude's ability to follow your documented patterns.

Start by asking Claude to explain how the authentication system works in the Todo API. Pay close attention — Claude should automatically use the Read tool to examine the relevant files without asking for your permission. This confirms that your auto-approve setting for Read is working correctly.

Next, test the permission controls for tools that make changes. Ask Claude to add a simple comment to one of the route files. Before Claude makes any edits, it should ask for your confirmation. This demonstrates that the Edit tool requires approval, just as you configured.

Finally, put the documentation to the test by requesting a small feature implementation. Ask Claude to add a new GET endpoint that returns todo statistics (such as the total count or the completed count). Watch how Claude implements this feature and verify that it follows the patterns documented in your CLAUDE.md:

    Uses async/await correctly
    Returns errors in the proper format
    Includes JSDoc comments
    Follows any middleware patterns you documented

By completing these tests, you'll confirm that your configuration creates a workspace where Claude works efficiently while respecting your guidelines and permissions!

To evaluate your workspace guardrails under real-world conditions, we will execute a series of tests to verify that your auto-approve read gates function seamlessly, your write permissions prompt for explicit authorization, and code generation aligns natively with your `CLAUDE.md` design patterns.

Execute this final step-by-step validation loop inside your interactive terminal session:

### Step 1: Verify the Auto-Approve Read Flow (Frictionless Path)

Type this analytical query into your `>` prompt and press **Enter**:

```text
Explain how the current authentication system works in the Todo API by looking into the middleware tracking files.

```

* **Verify the Trace:** Look closely at the top of the execution logs. The **`Read`** tool should fire instantly in the background without halting your session or printing a manual confirmation prompt. This proves your `allow` rule inside `.claude/settings.json` is perfectly operational.

---

### Step 2: Verify the Mutation Gatekeeper (The Security Prompt)

Now, test the enforcement of your modification constraints by issuing a minor file update directive. Type this request and press **Enter**:

```text
Add a simple inline comment to the top of src/routes/todos.js indicating its operational scope.

```

* **Verify the Trace:** Watch the execution pause! Instead of modifying your source files automatically, Claude Code must stop and display an interactive confirmation dialog box requesting your explicit authorization. This confirms that your `ask: ["Edit"]` constraint is securely blocking unverified file changes. Select **`1. Yes`** to let the patch finish.

---

### Step 3: Verify Architectural Alignment (Pattern Compliance Test)

To verify that Claude actively enforces your documented engineering patterns during code generation, command it to scaffold a new statistics route by pasting this prompt and pressing **Enter**:

```text
Implement a new GET endpoint at /api/todos/stats that calculates and returns basic todo statistics, such as total count and completed count.

```

### 🔍 Post-Generation Code Audit Checklist:

Once the generation diff completes, inspect the output code blocks to ensure Claude complied with your `CLAUDE.md` constitution across these four checkpoints:

* **Control Flow:** Confirm the endpoint is built entirely using `async/await` syntax instead of old-school callback trees.
* **Response Envelope:** Verify that the successful payload is cleanly packed within a root `{ data: {...} }` object.
* **Error Routing:** Ensure the entire logical block is nested inside a strict `try/catch` matrix that serializes exceptions directly into your standardized `{ error: error.message }` wire format.
* **Documentation:** Check that the route handler is preceded by a descriptive, well-formed **JSDoc** comment array.

---

### Step 4: Finalize and Submit

With your permission systems verified for safety and your generation tracks confirmed to be completely pattern-compliant, finalize and close out this practical configuration course module by typing:

```text
/submit

```

By completing this exercise, you have proven that your workspace configuration builds a highly predictable environment—allowing Claude Code to analyze your files efficiently while strictly respecting your design guidelines and authorization limits! 🚀